# SPL Top 500 Data Exploration (Solution)

# <span style="color:red;"> Solution </span>

## SPL Top 500 Data Exploration

<span style="color:green;"> [Exercise Without
Solutions](SPL-Top-500-Data-Exploration-Exercise.qmd) </span>

These exercises explore checkout data from the Seattle Public Library
for the Top 500 “Greatest” Novels — the novels most widely held in
libraries according to OCLC. For more context about the dataset, see the
[data essay](../index.qmd).

**Concepts covered:**

-   Groupby and aggregation (sum of checkouts)
-   Sorting and ranking (top N values)
-   Time series line plots (monthly checkouts over time)
-   String filtering (finding titles that contain a keyword)
-   Pivot tables
-   Correlation matrices and heatmaps — a correlation coefficient
    measures how closely two variables move together, ranging from -1
    (perfect inverse relationship) to 1 (perfect positive relationship),
    with 0 meaning no relationship

# Load the data

In [1]:
import pandas as pd
import plotly.express as px

spl_df = pd.read_csv("https://responsible-datasets-in-context.s3.us-west-2.amazonaws.com/top_500_spl_df.csv")
spl_df.head()

/tmp/ipykernel_8373/3933414714.py:4: DtypeWarning: Columns (12) have mixed types. Specify dtype option on import or set low_memory=False.
  spl_df = pd.read_csv("https://responsible-datasets-in-context.s3.us-west-2.amazonaws.com/top_500_spl_df.csv")

5 rows × 46 columns

# Exercise 1

Find the top 10 authors and top 10 books by total checkouts in the SPL
Top 500 dataset. Display them as styled tables.

Save the results as `top_authors` and `top_books`.

In [2]:
top_authors = spl_df.groupby('author')['checkouts'].sum().nlargest(10).reset_index()
top_authors.columns = ['Author', 'Total Checkouts']
top_authors.style.background_gradient(cmap='Blues')

In [3]:
top_books = spl_df.groupby('top_500_title')['checkouts'].sum().nlargest(10).reset_index()
top_books.columns = ['Title', 'Total Checkouts']
top_books.style.background_gradient(cmap='Greens')

Discuss/consider: Which authors and books are most popular at the
Seattle Public Library? Are there any surprises?

# Exercise 2

Create a time series line plot of monthly checkouts for “Pride and
Prejudice” over time.

Filter the data for “Pride and Prejudice”, group by year and month, and
plot the results.

In [4]:
pride_df = spl_df[spl_df['top_500_title'] == 'Pride and Prejudice']
pride_monthly = pride_df.groupby(['checkoutyear', 'checkoutmonth'])['checkouts'].sum().reset_index()
pride_monthly['date'] = pd.to_datetime(
    pride_monthly['checkoutyear'].astype(str) + '-' +
    pride_monthly['checkoutmonth'].astype(str) + '-01')

fig = px.line(pride_monthly.sort_values('date'), x='date', y='checkouts',
              title='Monthly SPL Checkouts for Pride and Prejudice')
fig.show()

Discuss/consider: What patterns do you notice in the checkout trends?
Are there any seasonal patterns or notable changes over time?

# Exercise 3

Calculate the correlation between the monthly checkout patterns of Harry
Potter books and display the results as a heatmap.

Filter for Harry Potter titles, pivot the data so each book is a column,
compute the correlation matrix, and visualize it.

In [5]:
hp_df = spl_df[spl_df['top_500_title'].str.contains('Harry Potter', case=False)]
hp_monthly = hp_df.groupby(['top_500_title', 'checkoutyear', 'checkoutmonth'])['checkouts'].sum().reset_index()
hp_monthly['date'] = pd.to_datetime(
    hp_monthly['checkoutyear'].astype(str) + '-' +
    hp_monthly['checkoutmonth'].astype(str) + '-01')

hp_pivot = hp_monthly.pivot_table(index='date', columns='top_500_title', values='checkouts', fill_value=0)
cor_matrix = hp_pivot.corr()

fig = px.imshow(cor_matrix, text_auto='.2f',
                title='Correlation Between Harry Potter Book Checkouts',
                color_continuous_scale='RdBu_r')
fig.show()

Discuss/consider: Which Harry Potter books have the most correlated
checkout patterns? What might explain these correlations?